# Real-Time Banking Risk & Fraud Intelligence
### AI-Powered Risk Scoring Engine & Generative Explanations

This notebook demonstrates the machine learning risk-scoring and conversational AI components of the **Real-Time Banking Risk & Fraud Intelligence Solution Accelerator** built on Microsoft Fabric.

#### Key Objectives:
1. **Load and Analyze** pre-seeded historical card transaction datasets.
2. **Train a Predictive Machine Learning Model** to classify suspicious transactions and predict fraud risk scores.
3. **Construct an Ensemble Scoring Engine** matching transactional behavior heuristics.
4. **Integrate Generative AI (OpenAI)** to provide plain-language risk explanations and autonomous decision recommendations.

In [1]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# OpenAI library
try:
    from openai import OpenAI
    OPENAI_SUPPORT = True
except ImportError:
    OPENAI_SUPPORT = False

print("Libraries imported successfully.")

StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 7, Finished, Available, Finished, False)

Libraries imported successfully.


## 1. Load & Inspect Historical Banking Data
We load the generated credit card fraud dataset (`data/historical_credit_card_fraud.csv`) containing 10,000 transactions.

In [2]:
df = pd.read_csv("abfss://SmartBankingRiskWorkspace@onelake.dfs.fabric.microsoft.com/BankingFraudLakehouse.Lakehouse/Files/historical_credit_card_fraud.csv")
print(f"Dataset loaded successfully. Shape: {df.shape}")
df.head(5)

StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 8, Finished, Available, Finished, False)

Dataset loaded successfully. Shape: (10000, 17)


,transaction_id,customer_id,account_number,customer_name,timestamp,amount,merchant,merchant_category,payment_channel,transaction_type,country,city,state,device_type,ip_address,risk_score,is_fraud
0,TXN-101047,CUST-1098,ACC-8616379926,Customer 98,2026-04-19 11:37:53,1051.98,Reliance Digital,Electronics,POS,Transfer,India,New Delhi,Delhi,Android Tablet,192.168.1.55,14,0
1,TXN-107124,CUST-1008,ACC-6756332150,Customer 8,2026-04-19 11:43:24,341.21,HDFC Home Loan,Financial Services,UPI,Purchase,India,Bengaluru,Karnataka,Samsung Galaxy S24,49.206.12.180,19,0
2,TXN-103958,CUST-1064,ACC-6110572428,Customer 64,2026-04-19 11:47:20,455.52,Swiggy Delivery,Food & Dining,UPI,Withdrawal,India,Hyderabad,Telangana,iPhone 15,103.45.21.92,12,0
3,TXN-105646,CUST-1090,ACC-4218981849,Customer 90,2026-04-19 11:50:11,29212.25,Offsite ATM Bangalore,ATM Cash Out,ATM,Withdrawal,India,Hyderabad,Telangana,Samsung Galaxy S24,192.168.1.248,76,1
4,TXN-102451,CUST-1218,ACC-4574027737,Customer 218,2026-04-19 11:50:35,607.84,Zomato Food,Food & Dining,UPI,Purchase,India,Ahmedabad,Gujarat,MacBook Pro,192.168.1.25,17,0


## 2. Exploratory Data Analysis (EDA)
Let's check the baseline fraud distribution, payment channels, and geographical risks.

In [3]:
print("=== Baseline Fraud Summary ===")
print(df['is_fraud'].value_counts())
print(f"Fraud percentage: {round(df['is_fraud'].mean() * 100, 2)}%\n")

print("=== Fraud by Payment Channel ===")
channel_summary = df.groupby('payment_channel').agg(
    TotalTransactions=('transaction_id', 'count'),
    FraudTransactions=('is_fraud', 'sum'),
    AverageAmount=('amount', 'mean')
)
channel_summary['FraudRatio%'] = round((channel_summary['FraudTransactions'] / channel_summary['TotalTransactions']) * 100, 2)
print(channel_summary.sort_values(by='FraudRatio%', ascending=False))

StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 9, Finished, Available, Finished, False)

=== Baseline Fraud Summary ===
is_fraud
0    9761
1     239
Name: count, dtype: int64
Fraud percentage: 2.39%

=== Fraud by Payment Channel ===
                  TotalTransactions  FraudTransactions  AverageAmount  \
payment_channel                                                         
ATM                            1284                 99   10785.318653   
Internet Banking               1248                 90   15264.232780   
POS                            1198                 30   11204.040058   
Mobile Banking                 1278                  5    7515.315900   
UPI                            4992                 15    3365.786168   

                  FraudRatio%  
payment_channel                
ATM                      7.71  
Internet Banking         7.21  
POS                      2.50  
Mobile Banking           0.39  
UPI                      0.30  


## 3. Train the Banking Risk Predictor Model
We train a **Random Forest Classifier** to analyze transaction properties (amount, payment channel, country, transaction type) and output fraud probabilities, which we use as dynamic risk scores.

In [4]:
# Preprocess categorical features using One-Hot Encoding
feature_cols = ['amount', 'payment_channel', 'transaction_type', 'country']
X_raw = df[feature_cols]
y = df['is_fraud']

# Perform dummy encoding
X = pd.get_dummies(X_raw, columns=['payment_channel', 'transaction_type', 'country'])

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features count: {X_train.shape[1]}")
print(f"Training dataset size: {X_train.shape[0]}")
print(f"Testing dataset size: {X_test.shape[0]}")

# Initialize and train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# --- OPTIMIZATION: Serialize and save the trained model to Lakehouse Files ---
import joblib
model_dir = "/lakehouse/default/Files/models"
model_path = f"{model_dir}/fraud_forest_model.pkl"
os.makedirs(model_dir, exist_ok=True)
joblib.dump(clf, model_path)
print(f"Model serialized and saved successfully at: {model_path}")

# Evaluate classifier
y_pred = clf.predict(X_test)
print("\n=== Model Evaluation Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 10, Finished, Available, Finished, False)

Training features count: 15
Training dataset size: 8000
Testing dataset size: 2000


Model serialized and saved successfully at: /lakehouse/default/Files/models/fraud_forest_model.pkl

=== Model Evaluation Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1952
           1       1.00      1.00      1.00        48

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

=== Confusion Matrix ===
[[1952    0]
 [   0   48]]


## 4. Ensemble Risk Scoring Logic
We create a dynamic scoring system that merges our trained ML probabilities with expert rules (velocity constraints, country changes, time profile).

In [5]:
def calculate_ensemble_risk(txn):
    """
    Ensembles machine learning model probability (clf) with rule-based heuristics.
    txn: dictionary of a transaction event
    Returns final_score (0-100) and list of reasons
    """
    reasons = []
    heuristic_score = 10  # Baseline risk
    
    # --- 1. HEURISTIC RULES ENGINE ---
    # Heuristic 1: High Value Spike
    amount = float(txn.get('amount', 0))
    if amount >= 200000:
        heuristic_score += 45
        reasons.append("High-value spike above threshold of Rs. 2,00,000")
    elif amount >= 50000:
        heuristic_score += 15
        reasons.append("Elevated transaction amount")
        
    # Heuristic 2: ATM Daily Cash-Out Limit Spike (Pattern 3)
    if txn.get('payment_channel') == "ATM" and amount >= 40000:
        heuristic_score += 45
        reasons.append("ATM cash withdrawal exceeds daily card limit of Rs. 40,000")
        
    # Heuristic 3: Geographical Risk (Pattern 2)
    if txn.get('country') != "India":
        heuristic_score += 40
        reasons.append(f"Foreign location mismatch: transaction initiated from {txn.get('country')}")
        
    # Heuristic 4: Suspicious IP / Proxy range (Pattern 4)
    if txn.get('ip_address') == "185.220.101.44":
        heuristic_score += 35
        reasons.append("Transaction routed through known malicious Tor IP subnet")
        
    # Heuristic 5: Time Profile
    try:
        dt = datetime.strptime(txn['timestamp'], "%Y-%m-%d %H:%M:%S")
        if dt.hour in [1, 2, 3, 4]:
            heuristic_score += 20
            reasons.append("Suspicious midnight transaction window (1:00 AM - 4:30 AM)")
    except Exception:
        pass
        
    # Heuristic 6: Payment Channel & Device Risk
    if txn.get('payment_channel') == "Internet Banking" and "Unrecognized" in txn.get('device_type', ''):
        heuristic_score += 25
        reasons.append("Internet banking accessed via unrecognized device hardware")
        
    # --- 2. MACHINE LEARNING ENGINE ---
    ml_score = 0
    if 'clf' in globals() and clf is not None:
        try:
            # Create a single-row DataFrame matching the model training structure
            txn_df = pd.DataFrame([{
                'amount': amount,
                'payment_channel': txn.get('payment_channel', 'UPI'),
                'transaction_type': txn.get('transaction_type', 'Purchase'),
                'country': txn.get('country', 'India')
            }])
            # One-hot encode categorical features
            txn_encoded = pd.get_dummies(txn_df)
            # Reindex to match trained features shape perfectly
            txn_encoded = txn_encoded.reindex(columns=clf.feature_names_in_, fill_value=0)
            
            # Predict fraud probability using trained model
            ml_prob = clf.predict_proba(txn_encoded)[0][1]
            ml_score = int(ml_prob * 100)
        except Exception as e:
            print(f"[ML Model Scoring Error]: {e}")
            ml_score = 0
            
    # --- 3. ENSEMBLE COMBINATOR ---
    if ml_score > 70:
        reasons.append(f"AI Random Forest Classifier flagged transaction as high probability fraud ({ml_score}%)")
        
    final_score = max(ml_score, heuristic_score)
    final_score = min(final_score, 99)
    
    return final_score, reasons

StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 11, Finished, Available, Finished, False)

## 5. AI Explanations & Actionable Decisions Pipeline
We define a function to query OpenAI to obtain detailed text analysis and recommendation strategies for flagged bank accounts.

In [6]:
def explain_risk_with_ai(txn, risk_score, reasons):
    """
    Generates natural language explanations. Supports live OpenAI and graceful dry-run mock fallbacks.
    """
    import os
    import json
    from datetime import datetime

    # 1. Retrieve config keys
    config_path = os.path.join("..", "..", "config", "accelerator-config.json")
    if not os.path.exists(config_path):
        config_path = os.path.join("config", "accelerator-config.json")
        
    openai_key = os.environ.get("OPENAI_API_KEY")
    if not openai_key or openai_key == "YOUR_OPENAI_API_KEY":
        for p in [".env", os.path.join("..", ".env"), os.path.join("..", "..", ".env"), os.path.join("..", "..", "..", ".env")]:
            if os.path.exists(p):
                try:
                    with open(p, "r") as ef:
                        for line in ef:
                            line = line.strip()
                            if line and not line.startswith("#") and "=" in line:
                                k, v = line.split("=", 1)
                                os.environ[k.strip()] = v.strip().strip('"').strip("'")
                except Exception:
                    pass
        openai_key = os.environ.get("OPENAI_API_KEY")
        
    if not openai_key or openai_key == "YOUR_OPENAI_API_KEY":
        if os.path.exists(config_path):
            try:
                with open(config_path, "r") as f:
                    config = json.load(f)
                    openai_key = config.get("openaiConfig", {}).get("apiKey")
            except Exception:
                pass

    # 2. Intelligent, highly-tailored mock fallback reasoner
    amount = float(txn.get('amount', 0))
    channel = txn.get('payment_channel', 'Unknown')
    country = txn.get('country', 'India')
    ip = txn.get('ip_address', '')
    device = txn.get('device_type', '')
    cust = txn.get('customer_name', 'Customer')
    city = txn.get('city', 'Unknown')
    
    threat_vector = "Unknown Anomaly"
    regulatory_impact = "RBI Circular on Customer Liability (Rule 2017/18) - Flagged for review."
    remediation_tier1 = "Initiate instant security hold on Card/UPI channel."
    remediation_tier2 = "Send automated push-verification SMS request to registered mobile."
    remediation_tier3 = "Log case in AML/CFT system for Suspicious Transaction Report (STR) review."
    
    if amount >= 200000:
        threat_vector = "High-Value Transaction Spike (POS/Internet Banking Limit Breach)"
        regulatory_impact = "RBI Circular DBR.No.Leg.BC.78/09.07.005/2017-18 limits customer liability if unauthorized transaction is reported within 3 days. High value necessitates immediate preventive freeze to mitigate bank liability."
        remediation_tier1 = "Suspend UPI, POS, and Net-Banking channels immediately. Lock account funds transfer capabilities."
        remediation_tier2 = "Initiate high-priority telephone verification call via the Relationship Manager."
        remediation_tier3 = "Report to Fraud FMC and log in Suspicious Transaction Registry."
    elif channel == "ATM" and amount >= 40000:
        threat_vector = "ATM Velocity & Cash-Out Burst (Daily Debit Limit Breach)"
        regulatory_impact = "High cash-out volumes at unorthodox hours suggest physical card compromise or ATM cloning skimming attacks."
        remediation_tier1 = "Temporary lockdown of debit card status. Restrict domestic and international ATM channels."
        remediation_tier2 = "Trigger instant in-app security alert and SMS warning to registered number."
        remediation_tier3 = "Analyze local ATM terminal DVR footage and review terminal health reports for physical skimmer anomalies."
    elif country != "India":
        threat_vector = "Foreign Geolocation Mismatch (Impossible Travel Speed Anomaly)"
        regulatory_impact = "Cross-border transaction velocity violates standard risk boundaries. Compliance requires instant card lock to prevent cross-border money laundering."
        remediation_tier1 = "Place global card lockdown. Decline all international POS, ATM, and E-commerce gateways."
        remediation_tier2 = "Send critical security email alert and interactive mobile app push-notification confirmation request."
        remediation_tier3 = "Mark card status as 'Compromised' and queue automatic reissue protocol."
    elif ip == "185.220.101.44":
        threat_vector = "Midnight Takeover via Malicious Proxy (Tor Node Access)"
        regulatory_impact = "Routing transaction authentication through a known Tor exit node IP violates standard corporate Cyber Security policy."
        remediation_tier1 = "Terminate active internet-banking login session. Suspend online banking login credentials."
        remediation_tier2 = "Require mandatory step-up Multi-Factor Authentication (MFA) and force credential reset."
        remediation_tier3 = "Flag IP subnet in Security Information and Event Management (SIEM) dashboard for broad threat analysis."

    reasons_bullet = "\n- ".join(reasons)
    mock_ai_explanation = (
        f"### 🛡️ AI Risk Scoring Insight & Decision Summary\n\n"
        f"**Transaction ID:** {txn['transaction_id']} | **Risk Score:** {risk_score}% | **Customer:** {cust}\n\n"
        f"#### 🔍 1. Threat Analysis Summary\n"
        f"- **Threat Vector:** {threat_vector}\n"
        f"- **Risk Indicators Flagged:**\n- {reasons_bullet}\n"
        f"- **Contextual Analysis:** Contextual home profile indicates customer lives in {city}, India. Initiating transaction from {country} via {channel} channel using {device} is a deviation from standard transaction behaviors.\n\n"
        f"#### 🏛️ 2. Regulatory & Compliance Impact\n"
        f"- {regulatory_impact}\n\n"
        f"#### 🚀 3. Structured Operational Action Plan\n"
        f"- **Tier 1 (Immediate / Automated):** {remediation_tier1}\n"
        f"- **Tier 2 (Direct Outreach / Step-up):** {remediation_tier2}\n"
        f"- **Tier 3 (Log / Compliance):** {remediation_tier3}"
    )

    if not openai_key or openai_key == "YOUR_OPENAI_API_KEY" or not OPENAI_SUPPORT:
        # Graceful dry-run fallback
        return mock_ai_explanation
        
    # Live OpenAI API Invocations
    try:
        client = OpenAI(api_key=openai_key)
        prompt = (
            f"You are a seasoned enterprise banking risk manager and senior fraud analyst at a top Indian bank (e.g., HDFC, ICICI).\n"
            f"Perform an expert diagnostic threat analysis on the following anomalous transaction and generate a structured markdown report.\n\n"
            f"### Anomalous Transaction Details:\n"
            f"- Transaction ID: {txn['transaction_id']}\n"
            f"- Customer Name: {txn['customer_name']}\n"
            f"- Home Location: {txn['city']}, India\n"
            f"- Channel Utilized: {txn['payment_channel']}\n"
            f"- Amount: Rs. {txn['amount']}\n"
            f"- Current Geolocation: {txn['country']}, {txn['city']}\n"
            f"- Accessing Device: {txn['device_type']}\n"
            f"- Source IP Address: {txn['ip_address']}\n"
            f"- Heuristic Anomalies Flagged: {', '.join(reasons)}\n"
            f"- Integrated System Risk Score: {risk_score}%/100\n\n"
            f"### Required Structured Report Format (Markdown):\n"
            f"### 🛡️ AI Risk Scoring Insight & Decision Summary\n\n"
            f"**Transaction ID:** {txn['transaction_id']} | **Risk Score:** {risk_score}% | **Customer:** {txn['customer_name']}\n\n"
            f"#### 🔍 1. Threat Analysis Summary\n"
            f"- **Threat Vector:** [Classify threat, e.g., High-Value Transaction Spike, Geolocation Velocity Anomaly, Cash-Out Burst, Account Takeover (ATO)]\n"
            f"- **Risk Indicators Flagged:** [Briefly describe the significance of each of the flagged anomalies: {', '.join(reasons)}]\n"
            f"- **Contextual Analysis:** [Analyze the deviation from the customer's typical profile (Home location: {txn['city']}, device: {txn['device_type']}) and why this specific combination of location, channel, and IP is highly suspicious]\n\n"
            f"#### 🏛️ 2. Regulatory & Compliance Impact\n"
            f"- [Cite relevant Indian banking regulatory guidelines, such as the RBI guidelines on customer liability for unauthorized electronic transactions (DBR.No.Leg.BC.78/09.07.005/2017-18) and AML compliance, explaining why a swift hold protects both the bank and the client]\n\n"
            f"#### 🚀 3. Structured Operational Action Plan\n"
            f"- **Tier 1 (Immediate / Automated):** [Describe specific technical steps the system should trigger automatically, e.g. locking cards, disabling Net-Banking API tokens, blocking device ID]\n"
            f"- **Tier 2 (Direct Outreach / Step-up):** [Specify step-up security protocols or outreach, e.g. mandatory MFA challenge, Relationship Manager phone call verification]\n"
            f"- **Tier 3 (Log / Compliance):** [Specify compliance procedures, e.g. logging to Suspicious Transaction Registry, reporting to FIU-IND or FMC, checking physical ATM health if skimming is suspected]\n\n"
            f"Keep your tone professional, authoritative, structured, and extremely concise."
        )
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are an expert banking risk compliance officer and lead fraud investigator. You output highly structured, professional, and analytical markdown reports with actionable banking steps."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"{mock_ai_explanation}\n(API Connection Warning: {e})"


StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 12, Finished, Available, Finished, False)

## 6. Real-Time KQL Database Ingestion & Scoring Loop
Instead of mock test scenarios, this production cell queries the **real-time streaming rows** directly from your live Fabric KQL Database (**BankingRiskDB**) using Spark's native Kusto connector, scores them, and writes the results back!

In [8]:
# ============================================================
# CORRECTED CELL 6: Real-Time KQL Scoring Loop  (v4 — Direct REST Write)
# Fixes:
#   1. WRITE-BACK via REST .ingest inline with COMMA separator
#      (the old pipe-separated version crammed all data into
#       the transaction_id column — KQL expects CSV format)
#   2. 19-column schema match (includes risk_score_1, is_fraud_1)
#   3. Deduplication: excludes transaction_ids already scored
#   4. Verification query after write confirms rows landed
# ============================================================

# 1. Define KQL Database details
kql_query_uri = os.environ.get("KQL_QUERY_URI", "https://trd-d2106gp5ufbuq1042c.z1.kusto.fabric.microsoft.com")
kql_db_name = "BankingRiskDB"

import os
import time
import json as _json
import math
import joblib
import requests
from datetime import datetime, timezone


# ── Helper: lightweight REST query (v2 API for data queries) ──
def kql_rest_query(query, access_token, uri=kql_query_uri, db=kql_db_name):
    """Run a KQL data query via v2 REST API."""
    url = f"{uri.rstrip('/')}/v2/rest/query"
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    resp = requests.post(url, json={"db": db, "csl": query}, headers=headers, timeout=30)
    if resp.status_code != 200:
        raise Exception(f"KQL REST error {resp.status_code}: {resp.text[:300]}")
    for frame in resp.json():
        if isinstance(frame, dict) and frame.get("TableKind") == "PrimaryResult":
            cols = [c["ColumnName"] for c in frame.get("Columns", [])]
            rows = frame.get("Rows", [])
            return cols, rows
    return [], []


# ── Helper: REST management command (v1 API for .commands) ──
def kql_mgmt(csl, access_token, uri=kql_query_uri, db=kql_db_name):
    """Run a KQL management (.command) via v1 REST API."""
    url = f"{uri.rstrip('/')}/v1/rest/mgmt"
    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    resp = requests.post(url, json={"db": db, "csl": csl}, headers=headers, timeout=60)
    return resp


# ── Helper: write scored rows via REST .ingest inline (SYNCHRONOUS) ──
def write_scored_rows_rest(scored_rows, access_token):
    """
    Write scored rows directly to KQL using .ingest inline command.
    Uses COMMA-separated CSV format (KQL's default for .ingest inline).
    Handles the 19-column schema including risk_score_1 and is_fraud_1.
    Processes in batches of 50 rows.
    """
    if not scored_rows:
        return 0

    # Column order MUST match the actual KQL table schema (19 columns)
    # The table has: 17 original + risk_score_1 (int) + is_fraud_1 (int)
    columns_17 = [
        "transaction_id", "customer_id", "account_number", "customer_name",
        "timestamp", "amount", "merchant", "merchant_category",
        "payment_channel", "transaction_type", "country", "city", "state",
        "device_type", "ip_address", "risk_score", "is_fraud"
    ]

    total_written = 0
    batch_size = 50

    for batch_start in range(0, len(scored_rows), batch_size):
        batch = scored_rows[batch_start : batch_start + batch_size]

        data_lines = []
        for row in batch:
            values = []
            for col in columns_17:
                val = row.get(col, "")
                if val is None:
                    val = ""
                val_str = str(val)
                # CSV escape: if value contains comma or quote, wrap in double-quotes
                if "," in val_str or '"' in val_str or "\n" in val_str:
                    val_str = '"' + val_str.replace('"', '""') + '"'
                values.append(val_str)

            # Append empty values for risk_score_1 and is_fraud_1 (extra columns)
            values.append("")  # risk_score_1
            values.append("")  # is_fraud_1

            data_lines.append(",".join(values))

        inline_data = "\n".join(data_lines)
        csl = f".ingest inline into table real_time_transactions <|\n{inline_data}"
        resp = kql_mgmt(csl, access_token)

        if resp.status_code == 200:
            total_written += len(batch)
            print(f"  Batch {batch_start//batch_size + 1}: wrote {len(batch)} rows OK")
        else:
            print(f"  Batch {batch_start//batch_size + 1} ERROR: "
                  f"HTTP {resp.status_code} - {resp.text[:200]}")

    return total_written


# ── Helper: Spark read with exponential backoff retry ──
def spark_kql_read_with_retry(query, access_token, max_retries=3):
    """Read from KQL via Spark connector with retry + backoff on ThrottleException."""
    for attempt in range(1, max_retries + 1):
        try:
            df = spark.read \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("accessToken", access_token) \
                .option("kustoCluster", kql_query_uri) \
                .option("kustoDatabase", kql_db_name) \
                .option("kustoQuery", query) \
                .option("clientRequestPropertiesAsJson", '{"Options":{"servertimeout":"00:02:00"}}') \
                .load()
            return df
        except Exception as e:
            err = str(e)
            if "ThrottleException" in err or "throttled" in err.lower():
                wait = 10 * (2 ** (attempt - 1))
                print(f"  KQL throttled (attempt {attempt}/{max_retries}). Waiting {wait}s...")
                time.sleep(wait)
                if attempt == max_retries:
                    raise
            else:
                raise


try:
    # 2. Load ML model
    model_path = "/lakehouse/default/Files/models/fraud_forest_model.pkl"
    if 'clf' not in globals():
        if os.path.exists(model_path):
            print("Loading pre-trained ML Model from Lakehouse Storage...")
            clf = joblib.load(model_path)
        else:
            print("Pre-trained model not found. Please run Section 3 first!")

    # 3. Get built-in Fabric access token
    accessToken = mssparkutils.credentials.getToken('kusto')

    # ──────────────────────────────────────────────────────────
    # 4. SMART UNSCORED CHECK — excludes already-scored IDs
    #    KQL is append-only: scored rows are APPENDED alongside
    #    the originals. We exclude transaction_ids that already
    #    have a scored copy (risk_score > 0).
    # ──────────────────────────────────────────────────────────
    print("Checking for truly unscored transactions...")
    check_query = """
    let scored_ids = real_time_transactions
        | where risk_score > 0
        | distinct transaction_id;
    real_time_transactions
    | where risk_score == 0
    | where transaction_id !in (scored_ids)
    | count
    """
    cols, rows = kql_rest_query(check_query, accessToken)
    recent_count = rows[0][0] if rows else 0
    print(f"Truly unscored rows (no scored copy exists): {recent_count}")

    if recent_count == 0:
        print("\nAll transactions are already scored! No new unscored rows found.")
        print("Start event_simulator.py to stream new transactions, then re-run this cell.")
        recent_txs = []
        high_risk_count = 0
        log_msg = "All transactions already scored. No new scoring performed."

    else:
        # 5. Fetch TRULY UNSCORED rows via Spark (with throttle retry)
        print(f"\nFetching {recent_count} truly unscored rows via Spark connector...")
        time.sleep(3)

        kql_query_unscored = """
        let scored_ids = real_time_transactions
            | where risk_score > 0
            | distinct transaction_id;
        real_time_transactions
        | where risk_score == 0
        | where transaction_id !in (scored_ids)
        | order by timestamp desc
        """
        kql_df = spark_kql_read_with_retry(kql_query_unscored, accessToken)
        recent_txs = [row.asDict() for row in kql_df.collect()]

        if len(recent_txs) == 0:
            print("\nNo new unscored transactions found. Everything is fully up to date!")
            log_msg = "No new unscored transactions found."
            high_risk_count = 0

        else:
            print(f"\nFetched {len(recent_txs)} unscored transactions from KQL.\n")
            print("Running AI scoring engine...\n")

            scored_rows = []
            high_risk_count = 0

            for txn in recent_txs:
                # Convert Spark Timestamp to ISO-8601 string
                if txn.get('timestamp') and hasattr(txn['timestamp'], 'strftime'):
                    txn['timestamp'] = txn['timestamp'].strftime('%Y-%m-%dT%H:%M:%S.000Z')

                score, reasons = calculate_ensemble_risk(txn)
                txn['risk_score'] = int(score)
                txn['is_fraud'] = int(1 if score > 80 else 0)
                scored_rows.append(txn)

                if score > 80:
                    high_risk_count += 1
                    print(f"ANOMALY DETECTED | ID: {txn.get('transaction_id')} | Customer: {txn.get('customer_name')} | Score: {score}/100")
                    print(f"Warnings: {reasons}\n")
                    explanation = explain_risk_with_ai(txn, score, reasons)
                    print(explanation)
                    print("=" * 60)

            if len(scored_rows) > 0:
                # ════════════════════════════════════════════════════════
                # WRITE BACK via REST .ingest inline (SYNCHRONOUS!)
                # Uses COMMA-separated CSV — NOT pipe-separated!
                # Rows appear IMMEDIATELY in the KQL table.
                # ════════════════════════════════════════════════════════
                print(f"\nWriting {len(scored_rows)} scored rows via REST .ingest inline...")
                written_count = write_scored_rows_rest(scored_rows, accessToken)

                if written_count == len(scored_rows):
                    print(f"\nSuccessfully wrote ALL {written_count} scored rows to KQL!")
                else:
                    print(f"\nWrote {written_count}/{len(scored_rows)} rows. Some batches may have failed.")

                # ── VERIFICATION: Confirm rows actually landed ──
                time.sleep(2)
                print("\nVerifying scored rows landed in KQL...")
                sample_ids = [r.get('transaction_id', '') for r in scored_rows[:5]]
                id_filter = ", ".join([f"'{tid}'" for tid in sample_ids])
                verify_query = f"""
                real_time_transactions
                | where transaction_id in ({id_filter})
                | where risk_score > 0
                | project transaction_id, customer_name, risk_score, is_fraud
                """
                v_cols, v_rows = kql_rest_query(verify_query, accessToken)
                if v_rows:
                    print(f"VERIFIED: Found {len(v_rows)} scored rows in KQL:")
                    for vr in v_rows[:5]:
                        print(f"   {vr[0]} | {vr[1]} | risk_score={vr[2]} | is_fraud={vr[3]}")
                else:
                    print("Scored rows not yet visible. They may take a moment to appear.")

                log_msg = (
                    f"Wrote {written_count} scored transactions to KQL via REST. "
                    f"High-risk anomalies flagged: {high_risk_count}"
                )
                print(f"\n{log_msg}")

                # ── DEDUP CLEANUP: Remove original unscored rows ──
                # KQL is append-only, so scored rows coexist with originals.
                # Rebuild table keeping only 1 row per transaction_id
                # (the one with the highest risk_score).
                print("\nCleaning up redundant rows (deduplication)...")
                try:
                    time.sleep(2)
                    # Step A: Create temp deduped table
                    dedup_csl = """.set-or-replace real_time_transactions_deduped <|
real_time_transactions
| sort by transaction_id, risk_score desc
| summarize take_any(*) by transaction_id
"""
                    r_dedup = kql_mgmt(dedup_csl, accessToken)
                    if r_dedup.status_code == 200:
                        time.sleep(1)
                        # Step B: Replace original from temp
                        replace_csl = """.set-or-replace real_time_transactions <|
real_time_transactions_deduped
"""
                        r_replace = kql_mgmt(replace_csl, accessToken)
                        if r_replace.status_code == 200:
                            # Step C: Drop temp
                            kql_mgmt(".drop table real_time_transactions_deduped ifexists", accessToken)
                            print("Deduplication complete! No redundant rows remain.")
                        else:
                            kql_mgmt(".drop table real_time_transactions_deduped ifexists", accessToken)
                            print(f"Dedup replace step failed: {r_replace.status_code}")
                    else:
                        print(f"Dedup temp table creation failed: {r_dedup.status_code}")
                except Exception as dedup_err:
                    print(f"Dedup cleanup warning (non-fatal): {dedup_err}")
            else:
                log_msg = "Scored rows evaluation empty. No new records written."

    # 6. Write run log via REST
    print("\nLogging notebook run statistics...")
    try:
        create_csl = (
            ".create-merge table scoring_runs_log "
            "(timestamp: string, records_read: long, records_scored: long, "
            "anomalies_flagged: long, log_message: string)"
        )
        kql_mgmt(create_csl, accessToken)

        now_ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.000Z')
        scored_count = len(recent_txs) if 'recent_txs' in locals() else 0
        anomaly_count = int(high_risk_count) if 'high_risk_count' in locals() else 0
        msg = log_msg if 'log_msg' in locals() else "Run completed"
        msg_escaped = msg.replace("'", "\\'").replace(",", " ")

        ingest_csl = (
            f".ingest inline into table scoring_runs_log <| "
            f"{now_ts},{recent_count},{scored_count},{anomaly_count},{msg_escaped}"
        )
        r = kql_mgmt(ingest_csl, accessToken)
        if r.status_code == 200:
            print("Saved scoring run log to KQL Database!")
        else:
            print(f"Log write status: {r.status_code} - {r.text[:100]}")

    except Exception as log_err:
        print(f"Failed to write execution log to KQL: {log_err}")

except Exception as e:
    print(f"Error in Real-Time KQL Database scoring loop: {e}")
    import traceback
    traceback.print_exc()
    print("Ensure the simulator is running and your KQL Database is accessible.")


StatementMeta(, 0344123d-6f62-4d77-abf8-1f2782b06347, 16, Finished, Available, Finished, False)

Checking active raw telemetry ingestion over the last hour...
Unscored raw rows in last 1h: 256

Fetching 256 unscored rows via Spark connector...

Successfully fetched 256 new unscored raw transactions from KQL.

Running AI scoring loop and generating real-time KQL database updates...

🚨 [REAL-TIME ANOMALY DETECTED] ID: TXN-854331 | Customer: Xiti Sen | Score: 82/100
Heuristic Warnings: ['Foreign location mismatch: transaction initiated from Nigeria', 'AI Random Forest Classifier flagged transaction as high probability fraud (82%)']


--- AI Explanation ---
### 🛡️ AI Risk Scoring Insight & Decision Summary

**Transaction ID:** TXN-854331 | **Risk Score:** 82% | **Customer:** Xiti Sen

#### 🔍 1. Threat Analysis Summary
- **Threat Vector:** Foreign Geolocation Mismatch (Impossible Travel Speed Anomaly)
- **Risk Indicators Flagged:**
- Foreign location mismatch: transaction initiated from Nigeria
- AI Random Forest Classifier flagged transaction as high probability fraud (82%)
- **Contex